#### Brutal Job Fit Analyzer

a simple prompt that takes in your qualifications and brutally assesses it for any given job description

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pylatexenc.latex2text import LatexNodes2Text  # Note: you will need to run %pip install pylatexenc or uv add pylatexenc


load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("no api key found")
elif not api_key.startswith("sk-proj-"):
    print("an API key was found, but it doesn't start sk-proj- so its the wrong key")
elif api_key.strip() != api_key:
    print("an API key was found, but it might have space/tab characters at the start or end -> remove them")
else:
    print("API key found")

openai = OpenAI()

# NOTE: you will need to alter this section if your resume isn't in LaTeX format!

def parse_latex_resume(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        latex_content = file.read()

    # grab only the actual resume body
    document_body = latex_content.split(r"\begin{document}")[1].split(r"\end{document}")[0]

    # convert LaTeX into normal text
    resume_text = LatexNodes2Text().latex_to_text(document_body)

    # remove large spacing gaps
    resume_text = "\n".join(
        line.strip()
        for line in resume_text.splitlines()
        if line.strip()
    )

    return resume_text


# replace this with your own .tex resume file
resume = parse_latex_resume("SampleResume.tex")

job_description = """
Robotics Software Engineering Intern -- RoboXYZ

RoboXYZ is a technology company looking for an intern to help build software
for intelligent robots that can move between simulation and real-world deployment.

What you'll work on:
- Develop and test software for simulated and physical robots.
- Help integrate perception, control, and robot software into one system.
- Work with engineers to test new robotics features in simulation before deploying them to hardware.
- Debug and improve software used in real-time robotic applications.

What we're looking for:
- Currently pursuing a degree in Robotics, Computer Science, Engineering, or a related field.
- Comfortable programming in C++ and Python.
- Experience with ROS2 or another robotics software framework.
- Some experience working with physical or simulated robots.
- Coursework or project experience in robotics, controls, machine learning, or computer vision.
- Familiarity with Linux and Git.

Nice to have:
- Experience with Gazebo, Webots, or another robotics simulator.
- Experience using OpenCV or PyTorch.
- Experience deploying software to embedded hardware such as Raspberry Pi or Jetson.
- Previous robotics projects, research, or internship experience.
"""

system_prompt = """
You are a brutal, honest, and direct hiring manager for the company described in the job description.

You will be given a candidate's resume and a job description.

Your job is to carefully compare the candidate's skills, experience, projects, and education against the requirements of the role.

Give a thorough evaluation that includes:
- how well the candidate matches the role
- important skills or experience that are missing
- any major concerns or weaknesses
- a final decision on whether you would move the candidate forward or reject them

Be realistic and do not exaggerate the candidate's qualifications.
Judge the candidate based on the information provided in the resume and job description.
Take into account the competition and industry standards expected from this role to make the final decision.

Adapt your expectations to the candidate's level of experience and the seniority of the role.

Send a concise, well articulated message. It should be brief, straight to the point, and skimmable.

Do not use high vocabulary or wordy language. Make it easy, simple, and straight to the point.

Respond in markdown. Do not wrap the response in a code block.
"""

user_prompt = """
Review the following resume and job description.
"""


def messages_for(resume, job_description):
    return [
        {"role": "system", "content": system_prompt},

        {"role": "user", "content": user_prompt
         + "\nResume:\n" + resume
         + "\nJob Description:\n" + job_description}
    ]


prompt_messages = messages_for(resume, job_description)


response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=prompt_messages
)


result = response.choices[0].message.content

display(Markdown(result))